In [ ]:
import numpy as np

class GetGroundTruth:
    """
    Class 2: Consumes the raw data streams from the Initialization pipeline,
    constructs the data-augmented matrix manifolds, and extracts the 
    nominal A and B matrices via linear least-squares regression.
    """
    def __init__(self, init_instance):
        self.plant = init_instance

    def compute_nominal_matrices(self):
        """
        Executes the data-augmented regression pipeline over the analytic 
        vector field snapshots to uncover the flat local linear grid.
        """
        # Fetch the active sensor and derivative streams from Class 1
        X, U, X_dot = self.plant.stream_sensor_data()
        
        # Data Augmentation: Stack State (X) and Control Input (U) into Matrix Omega
        # Dimensions: (3 x N_SAMPLES)
        Omega = np.vstack([X, U])
        
        # Execute Moore-Penrose Pseudoinverse to resolve: X_dot = [A | B] * Omega
        A_B_augmented = X_dot @ np.linalg.pinv(Omega)
        
        # Slice the augmented mapping into independent nominal matrices
        A_nominal = A_B_augmented[:, :2]
        B_nominal = A_B_augmented[:, 2:3]
        
        return A_nominal, B_nominal